# 실습 문제 - 104. Human-in-the-Loop

`# TODO` 표시된 부분을 채우며 위에서부터 셀 단위로 실행하세요. (모범답안: `practice_answer_104_HITL_Agent.ipynb`)

### 문제 요약

1. **edit로 결과 개수 바꾸기**: 승인 대기 중인 쿼리를 꺼내 `LIMIT`을 3으로 바꾼 뒤 `edit` 결정으로 실행
2. **reject 후 재시도 흐름**: 개인정보 조회 쿼리를 사유와 함께 `reject` → 모델이 다시 만든 쿼리를 `approve`

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## 준비: 모델 + SQL 도구 + HITL 에이전트 (본문과 동일)

In [ ]:
import os
from langchain.chat_models import init_chat_model

model = init_chat_model(
    os.environ["OPENAI_MODEL"],
    model_provider="openai",
    use_responses_api=True,        # Responses API 사용 (Chat Completions에서는 추론 모델의 도구 호출 불가)
    reasoning={"effort": "low"},   # 추론 강도 낮춤 → 실습 속도·비용 절감
)

# --- SQL 도구 3개 (본문 1장과 동일) ---
import sqlite3
from langchain.tools import tool

DB_PATH = "Chinook.db"


def connect_readonly() -> sqlite3.Connection:
    """DB를 읽기 전용(mode=ro)으로 연결 — 사람이 실수로 승인해도 데이터는 바뀌지 않도록 하는 안전장치"""
    return sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)


@tool
def sql_db_list_tables() -> str:
    """데이터베이스의 테이블 목록을 쉼표로 구분해 반환합니다."""
    con = connect_readonly()
    try:
        rows = con.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';"
        ).fetchall()
        return ", ".join(row[0] for row in rows)
    finally:
        con.close()


@tool
def sql_db_schema(table_names: str) -> str:
    """쉼표로 구분한 테이블 이름들을 받아 각 테이블의 스키마(CREATE 문)를 반환합니다. 예) Artist, Album"""
    con = connect_readonly()
    try:
        results = []
        for table in (name.strip() for name in table_names.split(",")):
            row = con.execute(
                "SELECT sql FROM sqlite_master WHERE type='table' AND name=?;", (table,)
            ).fetchone()
            results.append(row[0] if row else f"Error: '{table}' 테이블이 데이터베이스에 없습니다.")
        return "\n\n".join(results)
    finally:
        con.close()


@tool
def sql_db_query(query: str) -> str:
    """SQL 조회 쿼리를 실행하고 결과를 반환합니다. 에러가 나면 쿼리를 고쳐서 다시 시도하세요."""
    con = connect_readonly()
    try:
        return str(con.execute(query).fetchall())
    except Exception as e:
        return f"Error: {e}"
    finally:
        con.close()


tools = [sql_db_list_tables, sql_db_schema, sql_db_query]

# --- HITL 에이전트 (본문 3장과 동일) ---
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

system_prompt = """
당신은 SQLite 데이터베이스(Chinook)에 대한 질문에 답하는 에이전트입니다.
먼저 테이블 목록과 관련 테이블의 스키마를 확인한 뒤, 질문에 맞는 SQL 조회 쿼리를 작성해 실행하세요.
사용자가 개수를 지정하지 않으면 결과는 최대 5개로 제한하세요.
쿼리 실행이 사람에게 거부되면, 거부 사유를 반영해 새 쿼리를 작성하세요.
"""

agent = create_agent(
    model,
    tools,
    system_prompt=system_prompt,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "sql_db_list_tables": False,   # 읽기만 하는 도구 → 승인 없이 실행
                "sql_db_schema": False,
                "sql_db_query": {"allowed_decisions": ["approve", "edit", "reject"]},  # 실행 전 승인 필요
            },
            description_prefix="SQL 실행 승인 대기",
        ),
    ],
    checkpointer=InMemorySaver(),  # 멈춘 지점을 저장했다가 이어가기 위해 필수
)

# --- 출력 도우미 (본문과 동일) ---
def run(agent, payload, config):
    """에이전트를 실행하고, 승인 대기(interrupt)면 대기 중인 작업을, 끝났으면 최종 답변을 출력"""
    result = agent.invoke(payload, config)
    interrupts = result.get("__interrupt__")
    if interrupts:
        request = interrupts[0].value   # {"action_requests": [...], "review_configs": [...]}
        for action, review in zip(request["action_requests"], request["review_configs"]):
            print(f"⏸ 승인 대기: {action['name']}")
            print(f"   인자     : {action['args']}")
            print(f"   허용 결정: {review['allowed_decisions']}")
    else:
        print("✅ 최종 답변:", result["messages"][-1].text)
    return result

## 1. edit로 결과 개수 바꾸기

대기 중인 호출은 `result["__interrupt__"][0].value["action_requests"]`에 있습니다.
모델이 만든 쿼리를 꺼내 `LIMIT`만 바꾼 뒤, `edit` 결정의 `edited_action`으로 전달합니다.

In [ ]:
import re
from langgraph.types import Command

config = {"configurable": {"thread_id": "practice_edit_1"}}
result = run(agent, {"messages": [{"role": "user", "content": "판매 수량이 가장 많은 트랙을 알려줘"}]}, config)

In [ ]:
# 모델이 만든 쿼리를 꺼내 LIMIT 을 3으로 변경 (LIMIT 이 없으면 끝에 추가)
# TODO: 승인 대기 중인 첫 번째 작업(interrupt 값의 action_requests[0])을 꺼내세요.
action = ...
query = action["args"]["query"].strip().rstrip(";")
# TODO: query 의 LIMIT 숫자를 3으로 바꾼 edited_query 를 만드세요.
#       (re.sub 활용, LIMIT 이 없으면 끝에 " LIMIT 3" 추가)
edited_query = ...
print("수정한 쿼리:", edited_query)

result = run(
    agent,
    Command(resume={"decisions": [{
        # TODO: edit 결정을 작성하세요. (type 과 edited_action: 도구 이름 + 수정한 쿼리 인자)
        ...
    }]}),
    config,
)

## 2. reject 후 재시도 흐름

거부 사유(`message`)는 모델에게 전달되고, 모델은 사유를 반영한 새 쿼리로 다시 승인을 요청합니다.

In [ ]:
config = {"configurable": {"thread_id": "practice_reject_1"}}
result = run(agent, {"messages": [{"role": "user", "content": "직원들의 이름, 생년월일, 주소를 보여줘"}]}, config)

In [ ]:
result = run(
    agent,
    Command(resume={"decisions": [{
        # TODO: 거부(reject) 결정을 작성하세요. message 에 거부 사유를 넣으세요.
        #       사유: "생년월일과 주소는 개인정보라 조회할 수 없습니다. 이름과 직책만 보여주세요."
        ...
    }]}),
    config,
)

In [ ]:
# 모델이 다시 만든 쿼리를 확인하고 승인
if result.get("__interrupt__"):
    new_query = result["__interrupt__"][0].value["action_requests"][0]["args"]["query"]
    print("BirthDate/Address 포함 여부:", any(col in new_query for col in ("BirthDate", "Address")))
    # TODO: 모델이 다시 만든 쿼리를 승인(approve)으로 재개하세요.
    result = ...

## 확인 포인트

- 1번: `edit`로 바꾼 쿼리가 실행되므로 최종 답변에 트랙이 **3개**만 나옵니다. 모델은 자신이 만든 쿼리가 아니라 **사람이 고친 쿼리의 결과**로 답합니다.
- 2번: reject 후 두 번째 승인 대기의 쿼리에는 `BirthDate`, `Address`가 없습니다 — 거부 사유가 모델의 다음 행동을 바꾼 것입니다.
- 어느 경우든 **같은 `thread_id`의 config**로 재개해야 멈춘 대화가 이어집니다.